**Import Libraries**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

> Create Separate External Connection and Credentials for ADLS location(As we are using serverless compute)And Point that new Adls Credential to External Connection and in Azure storage account give the Add role assignment for Storage Blob Data Contributor

**Display the data from ADLS**

In [0]:
display(dbutils.fs.ls(
    "abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/"
))

path,name,size,modificationTime
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/paramaterized-source-data/,paramaterized-source-data/,0,1788810922000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/,raw-data/,0,1788805500000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/transformed-data/,transformed-data/,0,1788805522000


In [0]:
display(dbutils.fs.ls(
    "abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/"
))

path,name,size,modificationTime
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/athletes.csv,athletes.csv,418492,1788810658000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/coaches.csv,coaches.csv,16889,1788810679000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/entriesgender.csv,entriesgender.csv,1123,1788810697000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/medals.csv,medals.csv,2414,1788810716000
abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data/teams.csv,teams.csv,35270,1788810737000


**Storage Variable**

In [0]:
sv='abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/raw-data'

**Read the Data Files**

In [0]:
athletes = spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load(f"{sv}/athletes.csv")

coaches = spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load(f"{sv}/coaches.csv")

entriesgender = spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load(f"{sv}/entriesgender.csv")
            
medals = spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load(f"{sv}/medals.csv")

teams = spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load(f"{sv}/teams.csv")

In [0]:
athletes.display()
coaches.display()
entriesgender.display()
medals.display()
teams.display()

PersonName,Country,Discipline
AALERUD Katrine,Norway,Cycling Road
ABAD Nestor,Spain,Artistic Gymnastics
ABAGNALE Giovanni,Italy,Rowing
ABALDE Alberto,Spain,Basketball
ABALDE Tamara,Spain,Basketball
ABALO Luc,France,Handball
ABAROA Cesar,Chile,Rowing
ABASS Abobakr,Sudan,Swimming
ABBASALI Hamideh,Islamic Republic of Iran,Karate
ABBASOV Islam,Azerbaijan,Wrestling


Name,Country,Discipline,Event
ABDELMAGID Wael,Egypt,Football,null
ABE Junya,Japan,Volleyball,null
ABE Katsuhiko,Japan,Basketball,null
ADAMA Cherif,C�te d'Ivoire,Football,null
AGEBA Yuya,Japan,Volleyball,null
AIKMAN Siegfried Gottlieb,Japan,Hockey,Men
AL SAADI Kais,Germany,Hockey,Men
ALAMEDA Lonni,Canada,Baseball/Softball,Softball
ALEKNO Vladimir,Islamic Republic of Iran,Volleyball,Men
ALEKSEEV Alexey,ROC,Handball,Women


Discipline,Female,Male,Total
3x3 Basketball,32,32,64
Archery,64,64,128
Artistic Gymnastics,98,98,196
Artistic Swimming,105,0,105
Athletics,969,1072,2041
Badminton,86,87,173
Baseball/Softball,90,144,234
Basketball,144,144,288
Beach Volleyball,48,48,96
Boxing,102,187,289


Rank,TeamCountry,Gold,Silver,Bronze,Total,Rank by Total
1,United States of America,39,41,33,113,1
2,People's Republic of China,38,32,18,88,2
3,Japan,27,14,17,58,5
4,Great Britain,22,21,22,65,4
5,ROC,20,28,23,71,3
6,Australia,17,7,22,46,6
7,Netherlands,10,12,14,36,9
8,France,10,12,11,33,10
9,Germany,10,11,16,37,8
10,Italy,10,10,20,40,7


TeamName,Discipline,Country,Event
Belgium,3x3 Basketball,Belgium,Men
China,3x3 Basketball,People's Republic of China,Men
China,3x3 Basketball,People's Republic of China,Women
France,3x3 Basketball,France,Women
Italy,3x3 Basketball,Italy,Women
Japan,3x3 Basketball,Japan,Men
Japan,3x3 Basketball,Japan,Women
Latvia,3x3 Basketball,Latvia,Men
Mongolia,3x3 Basketball,Mongolia,Women
Netherlands,3x3 Basketball,Netherlands,Men


**Some Transformation and Updation**

In [0]:
entriesgender.printSchema()

root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



In [0]:
entriesgender = entriesgender.withColumn("Female",col("Female").cast(FloatType()))\
    .withColumn("Male",col("Male").cast(FloatType()))\
    .withColumn("Total",col("Total").cast(FloatType()))

In [0]:
entriesgender.printSchema()

root
 |-- Discipline: string (nullable = true)
 |-- Female: float (nullable = true)
 |-- Male: float (nullable = true)
 |-- Total: float (nullable = true)



**Find the top countries with the highest number of gold medals**

In [0]:
top_gold_medal_countries = medals.orderBy("Gold", ascending=False).select("TeamCountry","Gold").display()

TeamCountry,Gold
United States of America,39
People's Republic of China,38
Japan,27
Great Britain,22
ROC,20
Australia,17
Netherlands,10
Italy,10
France,10
Germany,10


**# Calculate the average number of entries by gender for each discipline**

In [0]:
average_entries_by_gender = entriesgender.withColumn(
    'Avg_Female', entriesgender['Female'] / entriesgender['Total']
).withColumn(
    'Avg_Male', entriesgender['Male'] / entriesgender['Total']
)
average_entries_by_gender.display()

Discipline,Female,Male,Total,Avg_Female,Avg_Male
3x3 Basketball,32.0,32.0,64.0,0.5,0.5
Archery,64.0,64.0,128.0,0.5,0.5
Artistic Gymnastics,98.0,98.0,196.0,0.5,0.5
Artistic Swimming,105.0,0.0,105.0,1.0,0.0
Athletics,969.0,1072.0,2041.0,0.4747672709456149,0.5252327290543851
Badminton,86.0,87.0,173.0,0.49710982658959535,0.5028901734104047
Baseball/Softball,90.0,144.0,234.0,0.38461538461538464,0.6153846153846154
Basketball,144.0,144.0,288.0,0.5,0.5
Beach Volleyball,48.0,48.0,96.0,0.5,0.5
Boxing,102.0,187.0,289.0,0.35294117647058826,0.6470588235294118


In [0]:
athletes = athletes \
    .withColumn("Name", trim(col("PersonName"))) \
    .withColumn("Nationality", upper(trim(col("Country")))) \
    .withColumn("Discipline", trim(col("Discipline"))) \
    .dropDuplicates() \
    .withColumn("processed_timestamp", current_timestamp())

**Remove records where important columns are null**

In [0]:
athletes.display()

PersonName,Country,Discipline,Name,Nationality,processed_timestamp
ABAD Nestor,Spain,Artistic Gymnastics,ABAD Nestor,SPAIN,2026-09-08T20:54:09.594Z
ABAROA Cesar,Chile,Rowing,ABAROA Cesar,CHILE,2026-09-08T20:54:09.594Z
ABDALLA Abubaker Haydar,Qatar,Athletics,ABDALLA Abubaker Haydar,QATAR,2026-09-08T20:54:09.594Z
ABDEL LATIF Radwa,Egypt,Shooting,ABDEL LATIF Radwa,EGYPT,2026-09-08T20:54:09.594Z
ABOUELKASSEM Alaaeldin,Egypt,Fencing,ABOUELKASSEM Alaaeldin,EGYPT,2026-09-08T20:54:09.594Z
ABRAHAM Tadesse,Switzerland,Athletics,ABRAHAM Tadesse,SWITZERLAND,2026-09-08T20:54:09.594Z
ADAMUS Bartlomiej Stejan,Poland,Weightlifting,ADAMUS Bartlomiej Stejan,POLAND,2026-09-08T20:54:09.594Z
ADESOKAN Dorcas Ajoke,Nigeria,Badminton,ADESOKAN Dorcas Ajoke,NIGERIA,2026-09-08T20:54:09.594Z
ADILKHANOVA Alina,Kazakhstan,Rhythmic Gymnastics,ADILKHANOVA Alina,KAZAKHSTAN,2026-09-08T20:54:09.594Z
AGHAEIHAJIAGHA Soraya,Islamic Republic of Iran,Badminton,AGHAEIHAJIAGHA Soraya,ISLAMIC REPUBLIC OF IRAN,2026-09-08T20:54:09.594Z


In [0]:
athletes = athletes.filter(
    col("name").isNotNull() &
    col("nationality").isNotNull()
)

In [0]:
athletes.display()
coaches.display()
entriesgender.display()
medals.display()
teams.display()

PersonName,Country,Discipline,Name,Nationality,processed_timestamp
ABDI Bashir,Belgium,Athletics,ABDI Bashir,BELGIUM,2026-09-08T20:54:24.065Z
ABICHA Mohammed,Morocco,Beach Volleyball,ABICHA Mohammed,MOROCCO,2026-09-08T20:54:24.065Z
ACETI Vladimir,Italy,Athletics,ACETI Vladimir,ITALY,2026-09-08T20:54:24.065Z
ADAMEK Klaudia,Poland,Athletics,ADAMEK Klaudia,POLAND,2026-09-08T20:54:24.065Z
AGUIRRE Unai,Spain,Water Polo,AGUIRRE Unai,SPAIN,2026-09-08T20:54:24.065Z
AKGUN Omer,Turkey,Shooting,AKGUN Omer,TURKEY,2026-09-08T20:54:24.065Z
AKMAL Nurul,Indonesia,Weightlifting,AKMAL Nurul,INDONESIA,2026-09-08T20:54:24.065Z
AL FAIHAN Abdulrahman,Kuwait,Shooting,AL FAIHAN Abdulrahman,KUWAIT,2026-09-08T20:54:24.065Z
ALBRIKAN Feras,Saudi Arabia,Football,ALBRIKAN Feras,SAUDI ARABIA,2026-09-08T20:54:24.065Z
ALEKSANDROVA Ekaterina,ROC,Tennis,ALEKSANDROVA Ekaterina,ROC,2026-09-08T20:54:24.065Z


Name,Country,Discipline,Event
ABDELMAGID Wael,Egypt,Football,null
ABE Junya,Japan,Volleyball,null
ABE Katsuhiko,Japan,Basketball,null
ADAMA Cherif,C�te d'Ivoire,Football,null
AGEBA Yuya,Japan,Volleyball,null
AIKMAN Siegfried Gottlieb,Japan,Hockey,Men
AL SAADI Kais,Germany,Hockey,Men
ALAMEDA Lonni,Canada,Baseball/Softball,Softball
ALEKNO Vladimir,Islamic Republic of Iran,Volleyball,Men
ALEKSEEV Alexey,ROC,Handball,Women


Discipline,Female,Male,Total
3x3 Basketball,32.0,32.0,64.0
Archery,64.0,64.0,128.0
Artistic Gymnastics,98.0,98.0,196.0
Artistic Swimming,105.0,0.0,105.0
Athletics,969.0,1072.0,2041.0
Badminton,86.0,87.0,173.0
Baseball/Softball,90.0,144.0,234.0
Basketball,144.0,144.0,288.0
Beach Volleyball,48.0,48.0,96.0
Boxing,102.0,187.0,289.0


Rank,TeamCountry,Gold,Silver,Bronze,Total,Rank by Total
1,United States of America,39,41,33,113,1
2,People's Republic of China,38,32,18,88,2
3,Japan,27,14,17,58,5
4,Great Britain,22,21,22,65,4
5,ROC,20,28,23,71,3
6,Australia,17,7,22,46,6
7,Netherlands,10,12,14,36,9
8,France,10,12,11,33,10
9,Germany,10,11,16,37,8
10,Italy,10,10,20,40,7


TeamName,Discipline,Country,Event
Belgium,3x3 Basketball,Belgium,Men
China,3x3 Basketball,People's Republic of China,Men
China,3x3 Basketball,People's Republic of China,Women
France,3x3 Basketball,France,Women
Italy,3x3 Basketball,Italy,Women
Japan,3x3 Basketball,Japan,Men
Japan,3x3 Basketball,Japan,Women
Latvia,3x3 Basketball,Latvia,Men
Mongolia,3x3 Basketball,Mongolia,Women
Netherlands,3x3 Basketball,Netherlands,Men


**Write the transformed files into transformed-data folder and save it as a parquet file**

In [0]:
base_path = "abfss://tokyo-olympic-data@olympicstorageaccanand.dfs.core.windows.net/transformed-data"

In [0]:
athletes.write.mode("overwrite").format("parquet").save(
    f"{base_path}/athletes"
)

coaches.write.mode("overwrite").format("parquet").save(
    f"{base_path}/coaches"
)

entriesgender.write.mode("overwrite").format("parquet").save(
    f"{base_path}/entriesgender"
)

medals.write.mode("overwrite").format("parquet").save(
    f"{base_path}/medals"
)

teams.write.mode("overwrite").format("parquet").save(
    f"{base_path}/teams"
)